# Exploratory analysis: bronze -> silver -> gold

Sanity-checks the pipeline output at each layer. Run the pipeline first:

```bash
python src/pipeline/01_bronze_ingest.py
python src/pipeline/02_silver_transform.py
python src/pipeline/03_gold_load.py
```

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src" / "pipeline"))
import config

pd.set_option("display.max_columns", None)

## Bronze: raw landing zone

In [ ]:
bronze_batches = sorted(config.BRONZE_DIR.glob("ingest_date=*/raw.parquet"))
bronze = pd.read_parquet(bronze_batches[-1])
print(bronze.shape)
bronze.head(3)

## Silver: tickets + resolutions

In [ ]:
tickets = pd.read_parquet(config.SILVER_DIR / "tickets.parquet")
resolutions = pd.read_parquet(config.SILVER_DIR / "resolutions.parquet")
print("tickets:", tickets.shape, " resolutions:", resolutions.shape)
tickets.head(3)

In [ ]:
resolutions.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tickets["category"].value_counts().plot.barh(ax=axes[0], title="Tickets by category")
tickets["channel"].value_counts().plot.bar(ax=axes[1], title="Tickets by channel")
tickets["priority"].value_counts().reindex(config.PRIORITY_VALUES).plot.bar(
    ax=axes[2], title="Tickets by priority"
)
plt.tight_layout()
plt.show()

In [ ]:
daily = tickets.set_index("created_at").resample("D").size()
daily.plot(figsize=(14, 3), title="Daily ticket volume (trailing 12 months)")
plt.show()

In [ ]:
print("Canonical KB entries:", tickets["kb_entry_id"].nunique(), "/ total tickets:", len(tickets))
tickets.groupby("kb_entry_id")["ticket_id"].count().sort_values(ascending=False).head(5).rename("tickets_sharing_this_kb_entry")

## Gold: warehouse views

In [ ]:
con = duckdb.connect(str(config.DUCKDB_PATH), read_only=True)
deflection = con.execute("SELECT * FROM v_deflection_rate_by_intent").df()
deflection

In [ ]:
deflection.set_index("intent")["deflection_rate"].sort_values().plot.barh(
    figsize=(6, 8), title="AI deflection rate by intent"
)
plt.axvline(0.5, color="gray", linestyle="--", linewidth=1)
plt.show()

In [ ]:
handle_time_csat = con.execute("SELECT * FROM v_handle_time_csat_by_category").df()
handle_time_csat

In [ ]:
volume = con.execute("SELECT * FROM v_daily_volume_by_channel").df()
volume.pivot(index="ticket_date", columns="channel", values="ticket_count").fillna(0).plot(
    figsize=(14, 4), title="Daily ticket volume by channel"
)
plt.show()
con.close()